# 🚀 TRAINING RADAR PULSE DEINTERLEAVING - SERVER LOCAL 2x RTX 2080

**Paper:** [Radar Pulse Deinterleaving with Transformer](https://arxiv.org/abs/2503.13476)

---

## ⚠️ TRƯỚC KHI BẮT ĐẦU:

1. **Giải nén data** trước (BƯỚC 1)
2. **Chạy từng cell theo thứ tự** (Shift+Enter)
3. **Thời gian:** Setup ~5 phút, Training ~3-8 giờ

---

## 🔧 **BƯỚC 1: Giải nén Data**

In [16]:
from pathlib import Path

REPO_DIR = Path("/home/fitmta/Trang55/deinterleaving")
data_dir = REPO_DIR / "data"
if (data_dir / "data").exists():
    data_dir = data_dir / "data"

!pip install -e -q
!pip install -r {REPO_DIR / "models_implementation" / "requirements.txt"} -q
!pip install tensorboard jupyter -q

print("✅ Done!")

ERROR: -q is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).
✅ Done!
✅ Done!


In [22]:
%pip install -q gdown

Note: you may need to restart the kernel to use updated packages.


In [26]:
import gdown
from pathlib import Path

REPO_DIR = Path("/home/fitmta/Trang55/deinterleaving")

file_id  = "1H9GBRsTcmHJBi5v2saDVrPD9ESbBGWUQ"
zip_path = REPO_DIR / "data_from_drive.zip"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    output=str(zip_path),
    quiet=False
)

print(f"✅ Đã tải: {zip_path}")
print(f"📦 Dung lượng: {zip_path.stat().st_size/1e9:.2f} GB")


Downloading...
From (original): https://drive.google.com/uc?id=1H9GBRsTcmHJBi5v2saDVrPD9ESbBGWUQ
From (redirected): https://drive.google.com/uc?id=1H9GBRsTcmHJBi5v2saDVrPD9ESbBGWUQ&confirm=t&uuid=8f9f4e80-36eb-4aa1-a4eb-4db08b67c446
To: /home/fitmta/Trang55/deinterleaving/data_from_drive.zip
100%|██████████| 3.92G/3.92G [20:10<00:00, 3.24MB/s]

✅ Đã tải: /home/fitmta/Trang55/deinterleaving/data_from_drive.zip
📦 Dung lượng: 3.92 GB


In [27]:
# 📦 Giải nén data zip vừa tải
import zipfile
from pathlib import Path

data_dir = REPO_DIR / "data"
if (data_dir / "data").exists():
    data_dir = data_dir / "data"

zip_path = REPO_DIR / "data_from_drive.zip"
if not zip_path.exists():
    raise FileNotFoundError(f"Không tìm thấy file zip: {zip_path}")

print(f"📂 Giải nén vào: {data_dir}")
data_dir.mkdir(parents=True, exist_ok=True)

try:
    with zipfile.ZipFile(zip_path, "r") as zf:
        bad = zf.testzip()
        if bad is not None:
            raise zipfile.BadZipFile(f"File zip bị lỗi tại: {bad}")
        zf.extractall(data_dir)
except zipfile.BadZipFile as e:
    raise RuntimeError(
        "❌ File tải về không phải zip hợp lệ. "
        "Hãy chạy lại cell tải về (Cell 4) để tải đúng file.\n"
        f"Chi tiết: {e}"
    )

# Nếu giải nén ra thêm 1 lớp thư mục con, dịch lên
subdirs = [d for d in data_dir.iterdir() if d.is_dir() and d.name not in ["train","validation","test"]]
if subdirs and not (data_dir / "train").exists():
    inner = subdirs[0]
    for d in inner.iterdir():
        d.rename(data_dir / d.name)
    inner.rmdir()

print("✅ Giải nén xong!")

📂 Giải nén vào: /home/fitmta/Trang55/deinterleaving/data


RuntimeError: ❌ File tải về không phải zip hợp lệ. Hãy chạy lại cell tải về (Cell 4) để tải đúng file.
Chi tiết: File is not a zip file

In [2]:
import subprocess
from pathlib import Path
import os

# ============== THAY ĐỔI ĐÂY ==============
REPO_DIR = Path("/home/fitmta/Trang55/deinterleaving")   # ← thư mục chứa repo
# ==========================================

# Tìm file zip trong repo
zip_files = list(REPO_DIR.glob("*.zip"))
data_dir  = REPO_DIR / "data"
if (data_dir / "data").exists():
    data_dir = data_dir / "data"

if data_dir.exists() and any(data_dir.glob("train/*.h5")):
    print("✅ Data đã giải nén rồi! Bỏ qua bước này.")
elif zip_files:
    zip_path = zip_files[0]
    print(f"📦 Tìm thấy: {zip_path.name}")
    print(f"📂 Giải nén vào: {data_dir}")
    data_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(zip_path), "-d", str(data_dir)], check=True)
    # Nếu giải nén ra thêm 1 lớp thư mục con, dịch lên
    subdirs = [d for d in data_dir.iterdir() if d.is_dir() and d.name not in ["train","validation","test"]]
    if subdirs and not (data_dir / "train").exists():
        inner = subdirs[0]
        for d in inner.iterdir():
            d.rename(data_dir / d.name)
        inner.rmdir()
    print("✅ Giải nén xong!")
else:
    print("❌ Không tìm thấy file .zip trong:", REPO_DIR)
    print("   Hãy copy file data.zip vào thư mục:", REPO_DIR)

# Kiểm tra
def count_files(path, pattern):
    return len(list(path.glob(pattern))) if path.exists() else 0

train_h5 = count_files(data_dir / "train", "*.h5")
val_h5   = count_files(data_dir / "validation", "*.h5")
test_h5  = count_files(data_dir / "test", "*.h5")
train_pt = count_files(data_dir / "train", "*.pt")

print(f"\n📊 Data location: {data_dir}")
print(f"   └─ train:      {train_h5} .h5 | {train_pt} .pt")
print(f"   └─ validation: {val_h5} .h5")
print(f"   └─ test:       {test_h5} .h5")

if train_h5 == 0 and train_pt > 0:
    print("\n⚠️  Train chỉ có .pt nên train.py không đọc được (cần .h5).")
    print("   ➜ Cần tải bộ train .h5 (HuggingFace) hoặc giải nén data.zip đúng định dạng.")

✅ Data đã giải nén rồi! Bỏ qua bước này.

📊 Data location: /home/fitmta/Trang55/deinterleaving/data/data
   └─ train:      2500 .h5 | 0 .pt
   └─ validation: 250 .h5
   └─ test:       250 .h5


## 🔧 **BƯỚC 2: Kiểm tra GPU**

In [3]:
!nvidia-smi

import torch
print(f"\n{'='*60}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"Số GPU:          {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB")

if not torch.cuda.is_available():
    raise RuntimeError("GPU không có! Kiểm tra CUDA driver.")

print(f"\n✅ GPU sẵn sàng!")
print(f"{'='*60}")

Fri Apr 24 16:06:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080        Off |   00000000:1A:00.0 Off |                  N/A |
| 23%   34C    P8              9W /  215W |    2613MiB /   8192MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📚 **BƯỚC 3: Install Dependencies**

In [4]:
import sys

# Cài challenge package
print("📦 Cài đặt challenge package...")
%pip install -e {REPO_DIR} -q

# Cài model dependencies
print("📦 Cài đặt model dependencies...")
%pip install -r {REPO_DIR}/models_implementation/requirements.txt -q
%pip install tensorboard jupyter -q

print("\n✅ Tất cả dependencies đã được cài đặt!")

📦 Cài đặt challenge package...
Note: you may need to restart the kernel to use updated packages.
📦 Cài đặt model dependencies...
Note: you may need to restart the kernel to use updated packages.
📦 Cài đặt model dependencies...
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.

✅ Tất cả dependencies đã được cài đặt!
Note: you may need to restart the kernel to use updated packages.

✅ Tất cả dependencies đã được cài đặt!


## ⚙️ **BƯỚC 4: Cấu hình Training**

Chọn config phù hợp:

| Config | Epochs | Thời gian (2x RTX 2080) | V-measure |
|--------|--------|-------------------------|-----------|
| quick    | 3  | ~1-2 giờ  | ~0.75 |
| standard | 8  | ~4-6 giờ  | ~0.88 |
| extended | 12 | ~8-10 giờ | ~0.89 |

In [7]:
import os
from pathlib import Path

# ============== CHỌN CONFIG ==============
TRAINING_CONFIG = "standard"  # Options: "quick", "standard", "extended"
GPU_IDS = "1"               # "0" = chỉ GPU 0 | "1" = chỉ GPU 1 | "0,1" = cả 2
# =========================================

configs = {
    "quick":    {"num_epochs": 3,  "validate_every": 1, "estimated_hours": "1-2"},
    "standard": {"num_epochs": 8,  "validate_every": 2, "estimated_hours": "4-6"},
    "extended": {"num_epochs": 12, "validate_every": 2, "estimated_hours": "8-10"},
}
config = configs[TRAINING_CONFIG]

output_dir = REPO_DIR / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Số worker tối ưu cho server
num_gpus   = len(GPU_IDS.split(","))
num_workers = 4 * num_gpus

print(f"{'='*60}")
print(f"📋 Training Configuration: {TRAINING_CONFIG.upper()}")
print(f"{'='*60}")
print(f"GPU:              {GPU_IDS} ({num_gpus} GPU)")
print(f"Epochs:           {config['num_epochs']}")
print(f"Batch size:       8  (4 per GPU khi dùng 2 GPU)")
print(f"Validate every:   {config['validate_every']} epochs")
print(f"Num workers:      {num_workers}")
print(f"Data:             {data_dir}")
print(f"Output:           {output_dir}")
print(f"Thời gian dự kiến: {config['estimated_hours']} giờ")
print(f"{'='*60}")

📋 Training Configuration: STANDARD
GPU:              1 (1 GPU)
Epochs:           8
Batch size:       8  (4 per GPU khi dùng 2 GPU)
Validate every:   2 epochs
Num workers:      4
Data:             /home/fitmta/Trang55/deinterleaving/data/data
Output:           /home/fitmta/Trang55/deinterleaving/outputs
Thời gian dự kiến: 4-6 giờ


## 🧪 **BƯỚC 5: Demo Training (subset 100k windows)** — *Tùy chọn*

Chạy nhanh để kiểm tra pipeline hoạt động đúng. Bỏ qua nếu muốn vào thẳng full training.

In [ ]:
import time, os

demo_output_dir = REPO_DIR / "outputs_demo_subset"

print("="*60)
print("🧪 DEMO TRAINING (subset 100k windows)")
print("="*60)

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS

start_time = time.time()

!python {REPO_DIR}/models_implementation/train.py \
    --data_dir {data_dir} \
    --output_dir {demo_output_dir} \
    --subset_size 100000 \
    --batch_size 8 \
    --num_epochs {config['num_epochs']} \
    --learning_rate 0.0001 \
    --window_length 1000 \
    --min_emitters 2 \
    --validate_every {config['validate_every']} \
    --save_every 1 \
    --num_workers {num_workers}

elapsed = (time.time() - start_time) / 3600
print("\n" + "="*60)
print("✅ DEMO TRAINING HOÀN THÀNH!")
print(f"⏱️  Thời gian: {elapsed:.2f} giờ")
print(f"💾 Checkpoints: {demo_output_dir}")
print("="*60)

## 🚀 **BƯỚC 6: BẮT ĐẦU FULL TRAINING**

⏰ **Thời gian:** 4-6 giờ (standard config trên 2x RTX 2080)

💡 **Tips:**
- Checkpoints tự động lưu mỗi epoch
- Nếu bị ngắt giữa chừng, dùng `--resume` để tiếp tục
- Xem log realtime: mở terminal chạy `tail -f training.log`

In [8]:
import time, os

print("\n" + "="*60)
print("🚀 BẮT ĐẦU FULL TRAINING")
print("="*60)
print(f"Data:    {data_dir}")
print(f"Output:  {output_dir}")
print(f"Epochs:  {config['num_epochs']}")
print(f"GPU:     {GPU_IDS}")
print(f"Dự kiến: {config['estimated_hours']} giờ")
print("="*60 + "\n")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS

start_time = time.time()

!python {REPO_DIR}/models_implementation/train.py \
    --data_dir {data_dir} \
    --output_dir {output_dir} \
    --batch_size 8 \
    --num_epochs {config['num_epochs']} \
    --learning_rate 0.0001 \
    --window_length 500 \
    --min_emitters 2 \
    --validate_every {config['validate_every']} \
    --save_every 1 \
    --num_workers {num_workers}

elapsed = (time.time() - start_time) / 3600

print("\n" + "="*60)
print("🎉 TRAINING HOÀN THÀNH!")
print("="*60)
print(f"⏱️  Thời gian thực tế: {elapsed:.2f} giờ")
print(f"💾 Checkpoints: {output_dir}")
print("="*60)


🚀 BẮT ĐẦU FULL TRAINING
Data:    /home/fitmta/Trang55/deinterleaving/data/data
Output:  /home/fitmta/Trang55/deinterleaving/outputs
Epochs:  8
GPU:     1
Dự kiến: 4-6 giờ

Using device: cuda
Loading datasets...
Analyzing files:   0%|                                 | 0/2500 [00:00<?, ?it/s]Using device: cuda
Loading datasets...
Processing files for windows: 100%|█████████| 250/250 [00:04<00:00, 52.58file/s]
Train dataset: 611816 samples
Validation dataset: 63738 samples
Creating model...
Processing files for windows: 100%|█████████| 250/250 [00:04<00:00, 52.58file/s]
Train dataset: 611816 samples
Validation dataset: 63738 samples
Creating model...
Total parameters: 10,524,168
Total parameters: 10,524,168

Starting training...

Epoch 1/8
Epoch 1:   0%|                                        | 0/76477 [00:00<?, ?it/s]
Starting training...

Epoch 1/8
Epoch 1:   0%|                                        | 0/76477 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/home/fitmta/Tr

## 🔄 **TIẾP TỤC TRAINING (nếu bị ngắt giữa chừng)**

In [ ]:
import glob, os

# Tìm checkpoint mới nhất
checkpoints = sorted(glob.glob(str(output_dir / "run_*/checkpoint_epoch_*.pt")))

if checkpoints:
    latest_ckpt = checkpoints[-1]
    print(f"✅ Checkpoint mới nhất: {latest_ckpt}")
    print("\nChạy cell dưới để tiếp tục:")
else:
    print("⚠️  Không tìm thấy checkpoint nào")
    latest_ckpt = ""

In [ ]:
import time, os

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS
start_time = time.time()

!python {REPO_DIR}/models_implementation/train.py \
    --data_dir {data_dir} \
    --output_dir {output_dir} \
    --batch_size 8 \
    --num_epochs {config['num_epochs']} \
    --learning_rate 0.0001 \
    --window_length 1000 \
    --min_emitters 2 \
    --validate_every {config['validate_every']} \
    --save_every 1 \
    --num_workers {num_workers} \
    --resume {latest_ckpt}

elapsed = (time.time() - start_time) / 3600
print(f"\n⏱️  Thời gian: {elapsed:.2f} giờ")

## 📊 **BƯỚC 7: Tìm Best Model**

In [ ]:
import glob, os

runs = sorted(glob.glob(str(output_dir / "run_*")))

if runs:
    latest_run = runs[-1]
    best_model = os.path.join(latest_run, "best_model.pt")

    print(f"{'='*60}")
    print("🏆 BEST MODEL")
    print(f"{'='*60}")

    if os.path.exists(best_model):
        print(f"✅ Model path: {best_model}")
        print(f"📁 Run dir:    {latest_run}")

        # Đọc config đã lưu
        import json
        config_file = os.path.join(latest_run, "config.json")
        if os.path.exists(config_file):
            with open(config_file) as f:
                saved_config = json.load(f)
            print("\n📋 Config đã dùng:")
            print(json.dumps(saved_config, indent=2))
    else:
        print("⚠️  best_model.pt chưa được tạo")
else:
    print("⚠️  Không tìm thấy training runs")

print(f"{'='*60}")

## 📈 **BƯỚC 8: Đánh giá Model**

Evaluate trên validation set.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS

if os.path.exists(best_model):
    print("📊 Đánh giá model trên validation set...\n")

    !python {REPO_DIR}/models_implementation/inference.py \
        --checkpoint {best_model} \
        --data_dir {data_dir} \
        --subset validation \
        --batch_size 8 \
        --save_results {latest_run}/validation_results.json

    print(f"\n✅ Kết quả đã lưu tại: {latest_run}/validation_results.json")
else:
    print("⚠️  Best model không tồn tại. Chạy training trước.")

## 📊 **BONUS: TensorBoard**

In [ ]:
%load_ext tensorboard

tb_logs = sorted(glob.glob(str(output_dir / "run_*/tensorboard")))

if tb_logs:
    latest_tb = tb_logs[-1]
    print(f"📊 TensorBoard logs: {latest_tb}")
    %tensorboard --logdir {latest_tb}
else:
    print("⚠️  Không tìm thấy TensorBoard logs")

---

## 🎯 **KẾT QUẢ MONG ĐỢI (2x RTX 2080)**

| Config | Epochs | Thời gian | V-measure |
|--------|--------|-----------|-----------|
| quick    | 3  | ~1-2 giờ  | ~0.75 |
| standard | 8  | ~4-6 giờ  | ~0.88 |
| extended | 12 | ~8-10 giờ | ~0.89 |

---

## 💡 **TROUBLESHOOTING**

### ❌ "CUDA Out of Memory"
```python
# Giảm batch_size trong BƯỚC 4 hoặc thêm vào lệnh train:
--batch_size 4
```

### ❌ Bị ngắt giữa chừng
Dùng cell **TIẾP TỤC TRAINING** ở trên, checkpoint đã lưu tự động.

### ❌ Muốn chạy nền (không cần giữ Jupyter mở)
```bash
# Trong terminal trên server:
nohup python train.py --data_dir ... --output_dir ... > training.log 2>&1 &
tail -f training.log
```